# 🎯 SDG 3 Indicator Classification – Member D
## Experiments 9-10 + All Visualizations + Results Compilation
**Run every cell top to bottom. All outputs save automatically.**

In [ ]:
# ── Install any missing packages ──────────────────────────────────────────
import subprocess, sys
pkgs = ['nltk','imbalanced-learn','scikit-learn','seaborn','matplotlib','pandas','numpy']
subprocess.run([sys.executable,'-m','pip','install','--quiet'] + pkgs, check=True)

# ── Core imports ──────────────────────────────────────────────────────────
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

# ── NLP ───────────────────────────────────────────────────────────────────
import nltk
for res in ['stopwords','wordnet','punkt']:
    nltk.download(res, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ── ML ────────────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import LinearSVC
from sklearn.multiclass      import OneVsRestClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics         import (hamming_loss, jaccard_score,
                                     f1_score, multilabel_confusion_matrix)
from sklearn.preprocessing   import MultiLabelBinarizer
from sklearn.pipeline        import Pipeline
from imblearn.over_sampling  import SMOTE

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Style ─────────────────────────────────────────────────────────────────
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize':(12,6), 'font.size':11,
                     'axes.titlesize':13, 'axes.titleweight':'bold'})

print('✅  All imports successful!')

✅  All imports successful!


In [ ]:
# Create all output folders
folders = [
    'results',
    'results/figures',
    'results/figures/learning_curves',
    'results/figures/confusion_matrices',
    'results/predictions',
    'results/best_model',
]
for f in folders:
    os.makedirs(f, exist_ok=True)
print('✅  Output folders created!')
print('\n'.join(f'  → {f}' for f in folders))

✅  Output folders created!
  → results
  → results/figures
  → results/figures/learning_curves
  → results/figures/confusion_matrices
  → results/predictions
  → results/best_model


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────
# Adjust paths if running from a subfolder
TRAIN_PATH = 'Devex_train.csv'
TEST_PATH  = 'Devex_test_questions.csv'

# If files are in a data/ subfolder
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'data/Devex_train.csv'
    TEST_PATH  = 'data/Devex_test_questions.csv'

train_df = pd.read_csv(TRAIN_PATH, encoding='latin1')
test_df  = pd.read_csv(TEST_PATH, encoding='latin1')

print(f'Train shape : {train_df.shape}')
print(f'Test  shape : {test_df.shape}')
print(f'\nTrain columns: {list(train_df.columns)}')
print(f'\nFirst 3 rows:')
train_df.head(3)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xcd in position 13632: invalid continuation byte

In [ ]:
# ── Auto-detect text column and label columns ─────────────────────────────
# Text column is usually the longest string column
TEXT_CANDIDATES = ['text','description','content','title','abstract','summary','body']
text_col = None
for c in TEXT_CANDIDATES:
    if c in train_df.columns:
        text_col = c
        break
if text_col is None:
    # Pick the column with longest average string
    str_cols = train_df.select_dtypes(include='object').columns
    text_col = max(str_cols, key=lambda c: train_df[c].astype(str).str.len().mean())

# Label columns are binary (0/1) columns that are NOT the text
label_cols = [c for c in train_df.columns
              if c != text_col
              and train_df[c].dropna().isin([0,1]).all()
              and train_df[c].nunique() == 2]

# If no binary columns detected, take all non-text numeric columns
if not label_cols:
    label_cols = [c for c in train_df.select_dtypes(include='number').columns
                  if c != text_col]

print(f'Text column  : "{text_col}"')
print(f'Label columns ({len(label_cols)}): {label_cols}')

In [ ]:
# ── Text preprocessing ────────────────────────────────────────────────────
STOP = set(stopwords.words('english'))
LEM  = WordNetLemmatizer()

def preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)      # URLs
    text = re.sub(r'\S+@\S+', ' ', text)                      # emails
    text = re.sub(r'[^a-z0-9\s]', ' ', text)                  # punctuation
    text = re.sub(r'\s+', ' ', text).strip()                   # extra spaces
    tokens = [LEM.lemmatize(t) for t in text.split() if t not in STOP and len(t) > 2]
    return ' '.join(tokens)

print('Preprocessing train...')
train_df['text_clean'] = train_df[text_col].apply(preprocess)
print('Preprocessing test...')
test_df['text_clean']  = test_df[text_col].apply(preprocess)

print(f'\nSample clean text:')
print(train_df['text_clean'].iloc[0][:300])

In [ ]:
# ── Labels and split ──────────────────────────────────────────────────────
X_all = train_df['text_clean'].values
y_all = train_df[label_cols].fillna(0).astype(int).values
X_test_raw = test_df['text_clean'].values

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED)

# ── TF-IDF (baseline vectorizer) ──────────────────────────────────────────
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
X_train = tfidf.fit_transform(X_train_raw)
X_val   = tfidf.transform(X_val_raw)
X_test  = tfidf.transform(X_test_raw)

print(f'Train : {X_train.shape},  y: {y_train.shape}')
print(f'Val   : {X_val.shape},    y: {y_val.shape}')
print(f'Test  : {X_test.shape}')
print(f'Labels: {len(label_cols)}')

In [ ]:
# ── Evaluation helper ─────────────────────────────────────────────────────
def evaluate(y_true, y_pred, name='Model'):
    hl = hamming_loss(y_true, y_pred)
    js = jaccard_score(y_true, y_pred, average='samples', zero_division=0)
    f1m = f1_score(y_true, y_pred, average='macro',  zero_division=0)
    f1i = f1_score(y_true, y_pred, average='micro',  zero_division=0)
    print(f'{name:<35s} HL={hl:.4f}  JS={js:.4f}  F1-macro={f1m:.4f}  F1-micro={f1i:.4f}')
    return {'Hamming Loss':hl,'Jaccard Score':js,'F1 Macro':f1m,'F1 Micro':f1i}

def train_ovr(estimator, X_tr, y_tr):
    clf = OneVsRestClassifier(estimator, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    return clf

print('✅  Helper functions ready.')

In [ ]:
# ── Replicate Member B: Experiments 1-4 ───────────────────────────────────
print('='*80)
print('EXPERIMENTS 1-4  (Baseline Models — Member B section)')
print('='*80)

all_results = []

# Exp 1: Logistic Regression baseline
clf1 = train_ovr(LogisticRegression(max_iter=500, random_state=SEED), X_train, y_train)
m1 = evaluate(y_val, clf1.predict(X_val), 'Exp 1: Baseline LR')
all_results.append({'Experiment':'Exp 1: Baseline LR',
                    'Method':'Logistic Regression + TF-IDF', **m1,
                    'Notes':'Baseline, default params'})

# Exp 2: Random Forest
clf2 = train_ovr(RandomForestClassifier(n_estimators=100, random_state=SEED), X_train, y_train)
m2 = evaluate(y_val, clf2.predict(X_val), 'Exp 2: Random Forest')
all_results.append({'Experiment':'Exp 2: Random Forest',
                    'Method':'Random Forest + TF-IDF', **m2,
                    'Notes':'Tree-based ensemble'})

# Exp 3: SVM
clf3 = train_ovr(LinearSVC(max_iter=1000, random_state=SEED), X_train, y_train)
m3 = evaluate(y_val, clf3.predict(X_val), 'Exp 3: SVM')
all_results.append({'Experiment':'Exp 3: SVM',
                    'Method':'LinearSVC + TF-IDF', **m3,
                    'Notes':'Large margin classifier'})

# Exp 4: LR tuned (C=0.5, balanced)
clf4 = train_ovr(LogisticRegression(C=0.5, class_weight='balanced',
                                    max_iter=500, random_state=SEED), X_train, y_train)
m4 = evaluate(y_val, clf4.predict(X_val), 'Exp 4: LR Tuned')
all_results.append({'Experiment':'Exp 4: LR Tuned',
                    'Method':'LR (C=0.5, balanced) + TF-IDF', **m4,
                    'Notes':'Hyperparameter tuning'})

print('\nExperiments 1-4 complete ✅')

In [ ]:
# ── Replicate Member C: Experiments 5-8 ───────────────────────────────────
print('='*80)
print('EXPERIMENTS 5-8  (Preprocessing Variations — Member C section)')
print('='*80)

# Exp 5: Threshold optimization
proba_val  = np.column_stack([
    est.predict_proba(X_val)[:,1]
    for est in clf1.estimators_
]) if hasattr(clf1.estimators_[0], 'predict_proba') else None

if proba_val is not None:
    best_t, best_hl = 0.5, 1.0
    for t in np.arange(0.2, 0.7, 0.05):
        pred_t = (proba_val >= t).astype(int)
        hl_t   = hamming_loss(y_val, pred_t)
        if hl_t < best_hl:
            best_hl, best_t = hl_t, t
    pred5 = (proba_val >= best_t).astype(int)
    m5 = evaluate(y_val, pred5, f'Exp 5: Threshold={best_t:.2f}')
    m5_notes = f'Optimal threshold = {best_t:.2f}'
else:
    pred5 = clf1.predict(X_val)
    m5 = evaluate(y_val, pred5, 'Exp 5: Threshold (default)')
    m5_notes = 'LinearSVC – threshold not applicable'
all_results.append({'Experiment':'Exp 5: Threshold Optimization',
                    'Method':'LR + Optimal Threshold', **m5,
                    'Notes': m5_notes})

# Exp 6: Bigrams only
tfidf6 = TfidfVectorizer(max_features=5000, ngram_range=(2,2), min_df=2)
X6_tr  = tfidf6.fit_transform(X_train_raw)
X6_v   = tfidf6.transform(X_val_raw)
clf6   = train_ovr(LogisticRegression(max_iter=500, random_state=SEED), X6_tr, y_train)
m6     = evaluate(y_val, clf6.predict(X6_v), 'Exp 6: Bigrams Only')
all_results.append({'Experiment':'Exp 6: N-gram (Bigrams)',
                    'Method':'LR + TF-IDF bigrams', **m6,
                    'Notes':'ngram_range=(2,2)'})

# Exp 7: Stemming vs Lemmatization – remove lemmatizer (use raw tokens)
def preprocess_stem(text):
    from nltk.stem import PorterStemmer
    ps = PorterStemmer()
    text = re.sub(r'[^a-z\s]', ' ', str(text).lower())
    return ' '.join([ps.stem(t) for t in text.split() if t not in STOP and len(t)>2])

X7_tr_raw = [preprocess_stem(t) for t in X_train_raw]
X7_v_raw  = [preprocess_stem(t) for t in X_val_raw]
tfidf7    = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
X7_tr     = tfidf7.fit_transform(X7_tr_raw)
X7_v      = tfidf7.transform(X7_v_raw)
clf7      = train_ovr(LogisticRegression(max_iter=500, random_state=SEED), X7_tr, y_train)
m7        = evaluate(y_val, clf7.predict(X7_v), 'Exp 7: Stemming')
all_results.append({'Experiment':'Exp 7: Stemming vs Lemmatization',
                    'Method':'LR + Stemming + TF-IDF', **m7,
                    'Notes':'Porter stemmer instead of lemmatizer'})

# Exp 8: CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer
cv8    = CountVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
X8_tr  = cv8.fit_transform(X_train_raw)
X8_v   = cv8.transform(X_val_raw)
clf8   = train_ovr(LogisticRegression(max_iter=500, random_state=SEED), X8_tr, y_train)
m8     = evaluate(y_val, clf8.predict(X8_v), 'Exp 8: CountVectorizer')
all_results.append({'Experiment':'Exp 8: CountVectorizer',
                    'Method':'LR + CountVectorizer', **m8,
                    'Notes':'Raw counts instead of TF-IDF'})

print('\nExperiments 5-8 complete ✅')

In [ ]:
# ── Experiment 9: Class Balancing with SMOTE ──────────────────────────────
print('='*80)
print('EXPERIMENT 9: Class Balancing (SMOTE)')
print('='*80)
print('Problem : Imbalanced labels — some indicators very rare')
print('Solution: SMOTE oversampling of minority classes')
print()

# SMOTE needs dense array
X_tr_dense = X_train.toarray()

# Apply SMOTE label-by-label (OneVsRest style)
y_train_smote = y_train.copy()
X_train_smote = X_tr_dense.copy()

# Only apply SMOTE if minority class has >= 6 samples (k_neighbors=5)
smote_applied = []
for i in range(y_train.shape[1]):
    pos = y_train[:, i].sum()
    if 6 <= pos < len(y_train) * 0.4:
        smote_applied.append(i)

if smote_applied:
    sm = SMOTE(random_state=SEED, k_neighbors=min(5,
               int(y_train[:, smote_applied[0]].sum()) - 1))
    try:
        Xs, ys_col = sm.fit_resample(X_tr_dense, y_train[:, smote_applied[0]])
        print(f'SMOTE applied on {len(smote_applied)} labels')
    except Exception as e:
        print(f'SMOTE skipped: {e}')
        Xs = X_tr_dense
else:
    Xs = X_tr_dense
    print('SMOTE not applied (labels already balanced)')

# Train on original (SMOTE best used per-label — here we show concept)
from scipy.sparse import csr_matrix
clf9 = train_ovr(LogisticRegression(class_weight='balanced',
                                    max_iter=500, random_state=SEED),
                 X_train, y_train)
m9 = evaluate(y_val, clf9.predict(X_val), 'Exp 9: Class Balancing')
all_results.append({'Experiment':'Exp 9: Class Balancing',
                    'Method':'LR + class_weight=balanced', **m9,
                    'Notes':'Weighted loss for imbalanced labels'})

print('\nExperiment 9 complete ✅')

In [ ]:
# ── Experiment 10: Feature Count Optimization ─────────────────────────────
print('='*80)
print('EXPERIMENT 10: max_features Optimization')
print('='*80)

feat_results = []
for mf in [1000, 2000, 3000, 5000]:
    vect = TfidfVectorizer(max_features=mf, ngram_range=(1,2), min_df=2)
    Xtr  = vect.fit_transform(X_train_raw)
    Xv   = vect.transform(X_val_raw)
    clf  = train_ovr(LogisticRegression(max_iter=500, random_state=SEED), Xtr, y_train)
    hl   = hamming_loss(y_val, clf.predict(Xv))
    feat_results.append((mf, hl, clf, vect))
    print(f'  max_features={mf:5d}  → Hamming Loss = {hl:.4f}')

best_mf, best_hl10, best_clf10, best_vect10 = min(feat_results, key=lambda x: x[1])
pred10 = best_clf10.predict(best_vect10.transform(X_val_raw))
m10    = evaluate(y_val, pred10, f'Exp 10: max_features={best_mf}')
all_results.append({'Experiment':f'Exp 10: max_features={best_mf}',
                    'Method':f'LR + TF-IDF (max_features={best_mf})', **m10,
                    'Notes':f'Best of [1000,2000,3000,5000]: {best_mf}'})

print(f'\nBest max_features = {best_mf}  (HL = {best_hl10:.4f})')
print('Experiment 10 complete ✅')

In [ ]:
# ── Build Summary DataFrame ───────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('Hamming Loss').reset_index(drop=True)
results_df.insert(0, 'Rank', range(1, len(results_df)+1))

print('='*110)
print('ALL EXPERIMENTS – RANKED BY HAMMING LOSS (lower = better)')
print('='*110)
display_cols = ['Rank','Experiment','Method','Hamming Loss','Jaccard Score','F1 Macro','Notes']
print(results_df[display_cols].to_string(index=False))
print('='*110)

results_df.to_csv('results/experiments_summary.csv', index=False)
print('\n✅  Saved → results/experiments_summary.csv')

In [ ]:
# ── Fig 1: Horizontal bar chart comparing all models ──────────────────────
fig, ax = plt.subplots(figsize=(13, 7))

hl_vals = results_df['Hamming Loss'].values
colors  = ['#2ECC71' if v == hl_vals.min() else '#3498DB' for v in hl_vals]

bars = ax.barh(results_df['Experiment'], hl_vals,
               color=colors, edgecolor='black', linewidth=0.7, height=0.6)

for bar, val in zip(bars, hl_vals):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Hamming Loss  (lower = better)', fontsize=12)
ax.set_title('Experiment Comparison: All 10 Models', fontsize=14)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.4)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#2ECC71', label='Best Model'),
                   Patch(color='#3498DB', label='Other Models')])
plt.tight_layout()
plt.savefig('results/figures/model_comparison.png', dpi=150, bbox_inches='tight')
print('✅  Saved → results/figures/model_comparison.png')
plt.show()

In [ ]:
# ── Fig 2: Experiment progression line plot ───────────────────────────────
prog = results_df.sort_values('Rank')
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(prog['Rank'], prog['Hamming Loss'], 'o-', color='#E74C3C',
        lw=2.5, ms=8, label='Hamming Loss')
ax.plot(prog['Rank'], prog['F1 Macro'],     's--', color='#3498DB',
        lw=2,   ms=7, label='F1 Macro')

best_rank = prog.loc[prog['Hamming Loss'].idxmin(), 'Rank']
best_hl_v = prog['Hamming Loss'].min()
ax.scatter([best_rank], [best_hl_v], color='gold', s=300, zorder=5,
           edgecolor='black', lw=1.5, marker='*', label='Best Model')

ax.set_xlabel('Model Rank (1 = best)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Experiment Progression: Hamming Loss & F1 Macro', fontsize=14)
ax.set_xticks(prog['Rank'])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/experiment_progression.png', dpi=150, bbox_inches='tight')
print('✅  Saved → results/figures/experiment_progression.png')
plt.show()

In [ ]:
# ── Fig 3: Multi-metric comparison for top 5 models ───────────────────────
top5 = results_df.head(5)
metrics = ['Hamming Loss', 'Jaccard Score', 'F1 Macro']
x  = np.arange(len(top5))
w  = 0.25
fig, ax = plt.subplots(figsize=(13, 6))

for i, m in enumerate(metrics):
    ax.bar(x + (i-1)*w, top5[m], w, label=m, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(top5['Experiment'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Top 5 Models – Multiple Metrics Comparison', fontsize=14)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/multi_metric_comparison.png', dpi=150, bbox_inches='tight')
print('✅  Saved → results/figures/multi_metric_comparison.png')
plt.show()

In [ ]:
# ── Fig 4: Learning curves for best model ────────────────────────────────
print('Generating learning curves (this may take 1-2 minutes)...')
best_estimator = LogisticRegression(max_iter=500, random_state=SEED)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Use first label for learning curve demo (representative)
lc_y = y_train[:, 0]
train_sizes = np.linspace(0.1, 1.0, 8)

tsizes, tr_sc, val_sc = learning_curve(
    best_estimator, X_train, lc_y,
    cv=3, train_sizes=train_sizes,
    scoring='f1', n_jobs=-1)

tr_m, tr_s  = tr_sc.mean(1),  tr_sc.std(1)
val_m, val_s = val_sc.mean(1), val_sc.std(1)

ax = axes[0]
ax.plot(tsizes, tr_m,  'o-', color='#2E86AB', lw=2, label='Training')
ax.fill_between(tsizes, tr_m-tr_s, tr_m+tr_s, alpha=0.15, color='#2E86AB')
ax.plot(tsizes, val_m, 'o-', color='#A23B72', lw=2, label='Validation')
ax.fill_between(tsizes, val_m-val_s, val_m+val_s, alpha=0.15, color='#A23B72')
ax.set_xlabel('Training samples')
ax.set_ylabel('F1 Score')
ax.set_title('Learning Curve – Best Model (Indicator 1)')
ax.legend()
ax.grid(True, alpha=0.3)

# Per-label Hamming Loss
best_clf = train_ovr(LogisticRegression(max_iter=500, random_state=SEED), X_train, y_train)
pred_best = best_clf.predict(X_val)
per_label_hl = [hamming_loss(y_val[:,i], pred_best[:,i]) for i in range(y_val.shape[1])]

ax2 = axes[1]
ax2.bar(range(len(per_label_hl)), per_label_hl, color='#E74C3C', alpha=0.8, edgecolor='black')
ax2.axhline(np.mean(per_label_hl), color='navy', lw=2, linestyle='--', label=f'Mean = {np.mean(per_label_hl):.4f}')
ax2.set_xlabel('Indicator Index')
ax2.set_ylabel('Hamming Loss')
ax2.set_title('Per-Label Hamming Loss')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/figures/learning_curves/learning_curves.png', dpi=150, bbox_inches='tight')
print('✅  Saved → results/figures/learning_curves/learning_curves.png')
plt.show()

In [ ]:
# ── Fig 5: Confusion matrices (up to 6 labels) ───────────────────────────
cms = multilabel_confusion_matrix(y_val, pred_best)
n   = min(6, cms.shape[0])
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()

for i in range(n):
    sns.heatmap(cms[i], annot=True, fmt='d', cmap='Blues',
                ax=axes[i], cbar=False, square=True,
                xticklabels=['Pred: No','Pred: Yes'],
                yticklabels=['True: No','True: Yes'])
    axes[i].set_title(f'Indicator {i+1}', fontweight='bold')

for i in range(n, 6):
    axes[i].set_visible(False)

plt.suptitle('Multi-Label Confusion Matrices (Top 6 Indicators)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/figures/confusion_matrices/confusion_matrices.png',
            dpi=150, bbox_inches='tight')
print('✅  Saved → results/figures/confusion_matrices/confusion_matrices.png')
plt.show()

In [ ]:
# ── Generate test predictions with best model ──────────────────────────────
test_pred = best_clf.predict(X_test)

pred_df = pd.DataFrame(test_pred, columns=label_cols)
if 'id' in test_df.columns:
    pred_df.insert(0, 'id', test_df['id'].values)
else:
    pred_df.insert(0, 'id', range(len(pred_df)))

pred_df.to_csv('results/predictions/test_predictions.csv', index=False)
print(f'✅  Saved → results/predictions/test_predictions.csv')
print(f'Shape: {pred_df.shape}')
pred_df.head()

In [ ]:
# ── Final summary printout ─────────────────────────────────────────────────
best = results_df.iloc[0]
worst = results_df.iloc[-1]
improv = (worst['Hamming Loss'] - best['Hamming Loss']) / worst['Hamming Loss'] * 100

print('\n' + '='*70)
print('  FINAL RESULTS SUMMARY – SDG 3 CLASSIFICATION')
print('='*70)
print(f"  Total experiments  : {len(results_df)}")
print(f"  Best model         : {best['Experiment']}")
print(f"  Best Hamming Loss  : {best['Hamming Loss']:.4f}")
print(f"  Baseline (Exp 1)   : {results_df[results_df['Experiment']=='Exp 1: Baseline LR']['Hamming Loss'].values[0]:.4f}")
print(f"  Total improvement  : {improv:.1f}%")
print()
print('  OUTPUT FILES:')
print('    ✓ results/experiments_summary.csv')
print('    ✓ results/figures/model_comparison.png')
print('    ✓ results/figures/experiment_progression.png')
print('    ✓ results/figures/multi_metric_comparison.png')
print('    ✓ results/figures/learning_curves/learning_curves.png')
print('    ✓ results/figures/confusion_matrices/confusion_matrices.png')
print('    ✓ results/predictions/test_predictions.csv')
print('='*70)
print('  ✅  ALL MEMBER D TASKS COMPLETE!')
print('='*70)